# SOAP UMAP plotting

Loads embedding parquets emitted by `analysis-soap-umap` and produces
static (matplotlib) and interactive (plotly) plots.

Inputs come from `analysis-soap-umap --output-prefix <name>`:
- `<name>.embedding.parquet` -- columns `structure_id`, `source`, `vertical`, `umap_x`, `umap_y` (and `umap_z` if `--n-components 3`)
- `<name>.png`               -- static preview
- `<name>.umap.joblib`       -- fitted reducer (only needed if you want `.transform()` of new vectors)

Run from the repo root so the relative paths below resolve.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

EMBED_ALL = Path("data/comparators/umap_all.embedding.parquet")
EMBED_OMOL = Path("data/comparators/umap_omol_by_vertical.embedding.parquet")

for p in (EMBED_ALL, EMBED_OMOL):
    print(p, "exists" if p.exists() else "missing")

## 1. Load and inspect

In [ ]:
def load_embedding(path: Path) -> pd.DataFrame:
    tbl = pq.read_table(str(path))
    df = tbl.to_pandas()
    md = tbl.schema.metadata or {}
    print(f"{path.name}: {len(df):,} rows")
    for k, v in md.items():
        print(f"  {k.decode()}: {v.decode()}")
    return df

df_all = load_embedding(EMBED_ALL)
df_omol = load_embedding(EMBED_OMOL)

In [ ]:
print("=== umap_all per (source, vertical) ===")
print(df_all.groupby(["source", "vertical"]).size().to_string())
print("\n=== umap_omol_by_vertical per vertical ===")
print(df_omol.groupby("vertical").size().sort_values(ascending=False).to_string())

## 2. Static matplotlib (mode=all, color by source)

In [ ]:
def scatter_2d(df, color_col, title, point_size=6, alpha=0.5, figsize=(11, 8)):
    fig, ax = plt.subplots(figsize=figsize, dpi=120)
    labels = sorted(df[color_col].unique())
    n = len(labels)
    if n <= 10:
        colors = [plt.get_cmap("tab10")(i) for i in range(n)]
    elif n <= 20:
        colors = [plt.get_cmap("tab20")(i) for i in range(n)]
    else:
        seq = []
        for cm in ("tab20", "tab20b", "tab20c"):
            cmap = plt.get_cmap(cm)
            seq.extend([cmap(i) for i in range(cmap.N)])
        colors = [seq[i % len(seq)] for i in range(n)]
    for c, lab in zip(colors, labels):
        sub = df[df[color_col] == lab]
        ax.scatter(sub.umap_x, sub.umap_y, s=point_size, alpha=alpha, color=c,
                   linewidths=0, label=f"{lab} (n={len(sub)})")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.set_title(title)
    if n > 20:
        ax.legend(bbox_to_anchor=(1.02, 0.5), loc="center left", fontsize=7, ncol=2, markerscale=3, framealpha=0.7)
    else:
        ax.legend(loc="best", fontsize=8, markerscale=3, framealpha=0.7)
    fig.tight_layout()
    return fig, ax

fig, _ = scatter_2d(df_all, "source", "SOAP UMAP -- OMol vs comparators")
plt.show()

## 3. Static matplotlib (omol-by-vertical, 33 classes)

In [ ]:
fig, _ = scatter_2d(df_omol, "vertical", "SOAP UMAP -- OMol per vertical", figsize=(13, 9))
plt.show()

## 4. Interactive plotly (2D)

Hover shows `structure_id`. Click legend entries to toggle classes.

In [ ]:
import plotly.express as px

fig = px.scatter(
    df_all,
    x="umap_x", y="umap_y",
    color="source",
    hover_name="structure_id",
    hover_data=["vertical"],
    opacity=0.6,
    title="SOAP UMAP -- mode=all (interactive)",
    width=900, height=700,
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [ ]:
fig = px.scatter(
    df_omol,
    x="umap_x", y="umap_y",
    color="vertical",
    hover_name="structure_id",
    opacity=0.6,
    title="SOAP UMAP -- omol-by-vertical (interactive)",
    width=1100, height=750,
)
fig.update_traces(marker=dict(size=3))
fig.show()

## 5. 3D (only if you reran with `--n-components 3`)

Re-run featurization with:

```bash
analysis-soap-umap \
  --input-dir data/comparators \
  --mode omol-by-vertical \
  --output-prefix data/comparators/umap_omol_by_vertical_3d \
  --n-per-class 500 \
  --n-components 3 --plot-3d-html
```

Then update the path below.

In [ ]:
EMBED_3D = Path("data/comparators/umap_omol_by_vertical_3d.embedding.parquet")
if EMBED_3D.exists():
    df_3d = pd.read_parquet(EMBED_3D)
    fig = px.scatter_3d(
        df_3d,
        x="umap_x", y="umap_y", z="umap_z",
        color="vertical",
        hover_name="structure_id",
        opacity=0.55,
        title="SOAP UMAP 3D",
        width=1000, height=750,
    )
    fig.update_traces(marker=dict(size=2))
    fig.show()
else:
    print(f"{EMBED_3D} not found -- re-run featurization with --n-components 3")

## 6. Filter / focus subsets

Easy way to declutter -- drop classes you don't care about, or zoom into a single source.

In [ ]:
drop = {"noble_gas", "noble_gas_compounds"}
df_omol_f = df_omol[~df_omol.vertical.isin(drop)]
fig, _ = scatter_2d(df_omol_f, "vertical",
                    f"SOAP UMAP -- omol verticals (excluded: {sorted(drop)})",
                    figsize=(13, 9))
plt.show()

In [ ]:
# Single-source overlay highlighting one vertical
highlight = "droplet"
fig, ax = plt.subplots(figsize=(11, 8), dpi=120)
bg = df_omol[df_omol.vertical != highlight]
fg = df_omol[df_omol.vertical == highlight]
ax.scatter(bg.umap_x, bg.umap_y, s=4, alpha=0.15, color="gray", linewidths=0, label="other")
ax.scatter(fg.umap_x, fg.umap_y, s=8, alpha=0.8, color="crimson", linewidths=0, label=highlight)
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
ax.set_title(f"OMol UMAP -- {highlight!r} on top of all others")
ax.legend(loc="best", markerscale=3, framealpha=0.8)
fig.tight_layout()
plt.show()

## 7. Project new SOAP vectors via the saved reducer

Use this when you want to add another comparator or a new OMol vertical to the same UMAP
*without* refitting. Point `EMBED_PARQUET` and `REDUCER` at matched files.

In [ ]:
import joblib

REDUCER = Path("data/comparators/umap_all.umap.joblib")
if REDUCER.exists():
    reducer = joblib.load(REDUCER)
    print("reducer hyperparams:")
    print(f"  metric={reducer.metric}, n_neighbors={reducer.n_neighbors}, min_dist={reducer.min_dist}")
    print(f"  fit shape: {reducer.embedding_.shape}")
    # Example: load a new SOAP parquet, transform.
    # NEW = pq.read_table('data/comparators/<somewhere>/soap.parquet', columns=['soap']).column('soap')
    # X_new = np.asarray(NEW.combine_chunks().values).reshape(-1, 32592)
    # Y_new = reducer.transform(X_new)
else:
    print(f"{REDUCER} not found")